# Ultralytics YOLO26 · AI Hub 알약 객체 탐지 (v2: SAM · SimCLR 스위치)

약 10,000장 / 118종 조합 데이터용 학습·평가·제출 파이프라인입니다. **RTX 5060(8GB, Blackwell) 기준**으로 정리했습니다.

- AI Hub partial JSON을 이미지 단위로 병합하고 `dl_mapping_code`를 클래스 라벨로 사용합니다.
- 같은 원본의 flip·색상 파생본이 train/valid에 섞이지 않도록 **augmentation family 단위로 분할**합니다(v2에서 실제 코드로 구현).
- 기본 설정: `yolo26s / 960px / batch 16 / AMP`. 화면에는 epoch마다 `Loss`와 대회 기준 `mAP@[0.75:0.95]`만 출력합니다.
- 추가 기능: `USE_SAM`(박스 정제), `USE_SIMCLR`(crop 분류기 재판정)을 설정 셀에서 True/False로 선택합니다.


## v2 추가 기능 요약

`USE_SAM`, `USE_SIMCLR` 스위치(설정 셀)로 아래 기능을 독립적으로 켜고 끕니다. 둘 다 `False`면 기존 파이프라인과 같은 결과를 냅니다.

| 스위치 | 하는 일 | 학습 데이터 |
|---|---|---|
| `USE_SAM` | 검출 박스를 SAM box prompt로 넣어 마스크 외접 박스로 정제 (학습 없음) | 없음 |
| `USE_SIMCLR` | train GT crop으로 SimCLR 대조학습 → 118종 분류기 → 검출 클래스 재판정·융합 | train split만 |

- 실행 순서: 검출기 학습 → best 로드 → **val 검출 캐시 → SAM → SimCLR → 변형 비교표 → 제출**
- 비교표는 `base / sam / simclr / sam+simclr` 중 켜진 조합만 val `mAP@[.75:.95]`로 계산합니다.
- SAM·SimCLR 준비가 실패하면 오류를 출력한 뒤 해당 기능만 끄고 계속 진행합니다.
- `SUBMIT_USE_BEST_VARIANT=True`면 스위치와 무관하게 val 점수가 가장 높은 변형으로 제출합니다.
- SAM 박스 padding, SimCLR 융합 비율은 val로 스칼라 1개씩 자동 튜닝합니다(`SAM_TUNE_PAD`, `SIMCLR_TUNE_ALPHA`).


## 0. 설치

### CUDA PyTorch 확인·복구

RTX 5060(sm_120)은 **CUDA 12.8 이상(cu128) · PyTorch 2.7 이상**이 필요합니다.

- `torch.cuda.is_available()`가 `True`이고 imports 셀의 GPU 호환성 검사를 통과하면 복구 셀은 건너뜁니다.
- 그렇지 않으면 복구 셀의 두 줄을 실행한 뒤 **커널을 재시작**하세요.
- 설치 셀은 Ultralytics를 최신으로 올립니다(YOLO26과 SAM2 지원에 필요). 기존 torch 버전은 바꾸지 않습니다.


In [ ]:
# CUDA가 False이거나 imports 셀에서 "sm_120 미지원" 오류가 날 때만 아래 두 줄의 주석을 풀어 실행하세요.
# 실행 후: Kernel Restart → 다음 셀부터 재실행
# %pip uninstall -y torch torchvision torchaudio
# %pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128


In [ ]:
# CUDA PyTorch가 준비된 뒤 한 번만 실행하세요. (torch는 건드리지 않습니다)
%pip install -q -U ultralytics
%pip install -q --upgrade-strategy only-if-needed kagglehub iterative-stratification "torchmetrics[detection]" faster-coco-eval supervision pandas matplotlib pyyaml wandb tqdm
# 설치 후 Kernel Restart → 아래 imports 셀부터 실행하세요.


## 1. imports · 재현성 · 하드웨어 확인


In [ ]:
import contextlib
import gc
import logging
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
import yaml
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
from ultralytics import YOLO, settings
import wandb

# Ultralytics의 자동 W&B 실행만 끔.
# 아래에서 wandb.init()으로 직접 기록한다.
settings.update({"wandb": False})

SEED = 42
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA PyTorch를 찾지 못했습니다. CPU 학습으로 진행하지 않습니다.\n"
        f"torch={torch.__version__}, compiled CUDA={torch.version.cuda}.\n"
        "설치 섹션의 CUDA PyTorch 복구 셀을 실행한 뒤 커널을 재시작하세요."
    )
def check_gpu_compat():
    """RTX 50 시리즈(Blackwell, sm_120)는 CUDA 12.8 이상 빌드의 PyTorch가 필요합니다."""
    if not torch.cuda.is_available():
        return
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 12 and f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        raise RuntimeError(
            f"이 PyTorch({torch.__version__}, CUDA {torch.version.cuda})는 GPU 아키텍처 sm_{major}{minor}를 지원하지 않습니다.\n"
            "설치 섹션의 cu128 복구 셀을 실행한 뒤 커널을 재시작하세요."
        )

check_gpu_compat()

gpu = torch.cuda.get_device_properties(0)
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU    : {gpu.name}")
print(f"VRAM   : {gpu.total_memory / 1024**3:.1f} GiB")
print("AMP    : 설정 셀의 AMP 값을 따릅니다 (RTX 5060은 AMP 사용 권장)")


## 2. 실험 설정

RTX 5060 8GB 기준: `yolo26s.pt / 960px / batch 4 / AMP`. `NBS=64`이므로 gradient accumulation이 자동 적용됩니다.
OOM이면 `BATCH=8`로 낮추고, 여유가 있으면 `yolo26m.pt`를 시험하세요.


In [ ]:
# W&B 로그인은 학습 시작 셀(11장)에서 USE_WANDB=True일 때만 수행합니다.
# (v2: 설정 셀보다 앞에서 USE_WANDB를 참조하던 순서 문제를 없앰)


In [ ]:
EXPECTED_NUM_CLASSES = 118
EXPECTED_TEST_IMAGES = 842

VAL_RATIO = 0.20
N_SPLIT_CANDIDATES = 200
GROUP_GEOMETRY_DECIMALS = 3
FAMILY_STRICT = False     # True: family 탐지가 이상하면 중단 / False: 경고만 출력하고 진행

MODEL_WEIGHTS = "yolo26s.pt"  # Ultralytics COCO pretrained checkpoint
IMGSZ = 960
BATCH = 4               # RTX 5060 8GB 기준. OOM이면 2로만 낮추고 IMGSZ는 유지
NBS = 64                 # nominal batch size; BATCH < NBS이면 gradient accumulation 적용
EPOCHS = 30
DETERMINISTIC = False    # True는 재현성↑ 속도↓ (일부 연산에서 오류 가능)

# W&B 설정
USE_WANDB = False # W&B 권한 오류가 학습을 막지 않게 기본은 로컬 기록
WANDB_PROJECT = "beginner-team-project"
WANDB_ENTITY = "jaedong0817-"  # 본인 W&B 계정에 기록. 팀 workspace 권한이 있으면 해당 entity 이름으로 변경

# Windows/Jupyter에서는 DataLoader worker가 멈추는 일이 잦아 0이 안전합니다. (안정적이면 4로 올려 보세요)
IS_WINDOWS = platform.system() == "Windows"
NUM_WORKERS = 0 if IS_WINDOWS else min(8, os.cpu_count() or 1)
AMP = True               # RTX 5060 Blackwell Tensor Core 기준 True
QUIET_EPOCH_LOG = True   # 각 epoch의 Loss / mAP@[.75:.95] 요약만 출력

RUN_SPEED_TEST = False   # True면 약 50 iteration만 1 epoch 실행 (실제 학습 전 권장)
RUN_TRAINING = True
RESUME = False           # True일 때 RUN_TRAINING은 False로 변경
EXISTING_RUN_DIR = None  # RUN_TRAINING=False로 기존 결과만 평가할 때 예: r"beginner-team-project/yolo26s_pills_960_b16_e30_family_v2"
if RUN_TRAINING and RESUME:
    raise ValueError("RUN_TRAINING과 RESUME은 동시에 True일 수 없습니다.")

PROJECT_DIR = Path("beginner-team-project")
MODEL_TAG = Path(MODEL_WEIGHTS).stem
RUN_NAME = f"{MODEL_TAG}_pills_{IMGSZ}_b{BATCH}_e{EPOCHS}_rtx5060"

OUTPUT_DIR = PROJECT_DIR / RUN_NAME
WANDB_RUN_NAME = RUN_NAME
PRED_BATCH_SIZE = 4      # 학습 종료 뒤 VRAM을 비웠어도 8GB에서 안정적으로 추론
PRED_CONFIDENCE = 0.001
EVAL_MAX_DETECTIONS = 20
SUBMISSION_MAX_DETECTIONS = 8
COMPETITION_IOUS = [0.75, 0.80, 0.85, 0.90, 0.95]
SUBMISSION_TAG = "yolo26"   # 제출 파일명 접두어

# ══════════════════════════════════════════════════════════════
# 추가 기능 스위치 (v2)
# ══════════════════════════════════════════════════════════════
USE_SAM = False      # True: SAM으로 검출 박스를 마스크 외접 박스로 정제
USE_SIMCLR = False   # True: SimCLR 사전학습 crop 분류기로 클래스 재판정 · 융합
SUBMIT_USE_BEST_VARIANT = False  # True: 스위치와 무관하게 val 점수가 가장 높은 변형으로 제출

# ── SAM 세부 설정 ─────────────────────────────────────────────
SAM_MODEL_CANDIDATES = ["sam2.1_b.pt", "sam2.1_l.pt", "sam_b.pt", "mobile_sam.pt"]  # 앞에서부터 로드 시도
SAM_TOPK = 8                # 이미지당 상위 몇 개 검출을 정제할지
SAM_MIN_SCORE = 0.01        # 이 점수 미만 검출은 정제하지 않음
SAM_MIN_IOU = 0.60          # 원본 박스와 IoU가 이보다 낮은 SAM 결과는 버리고 원본 유지
SAM_MIN_AREA_RATIO = 0.30   # 마스크 면적 / 박스 면적이 이보다 작으면 버림
SAM_BLEND = 1.0             # 1.0 = SAM 박스로 완전 교체, 0.5 = 원본과 평균
SAM_PAD_PX = 0.0            # 마스크 박스를 사방으로 넓히는 픽셀 수(GT 박스가 느슨하면 양수가 유리)
SAM_TUNE_PAD = True         # True: val에서 SAM_PAD_GRID를 비교해 SAM_PAD_PX 자동 선택
SAM_PAD_GRID = (-2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 6.0)

# ── SimCLR 세부 설정 (GT crop → 대조학습 → 118종 분류기 → 검출 클래스 재판정) ──
SIMCLR_BACKBONE = "resnet50"      # resnet18 / resnet34 / resnet50
SIMCLR_IMAGENET_INIT = True       # True: ImageNet 가중치에서 시작(다운로드 실패 시 자동으로 랜덤 초기화)
SIMCLR_CROP = 160                 # crop 한 변(px)
SIMCLR_CROP_PAD = 0.15            # 박스 주변 여백 비율
SIMCLR_EPOCHS = 30                # 대조학습 epoch (0이면 대조학습 생략, 분류 fine-tuning만)
SIMCLR_BATCH = 128                # OOM이면 자동으로 절반씩 줄여 재시도
SIMCLR_LR = 5e-4
SIMCLR_TEMP = 0.2
SIMCLR_CLS_EPOCHS = 12            # 118종 분류 fine-tuning epoch
SIMCLR_CLS_LR_BACKBONE = 2e-4
SIMCLR_CLS_LR_HEAD = 2e-3
SIMCLR_TOPK = 8                   # 이미지당 상위 몇 개 검출에 분류기를 적용할지
SIMCLR_MIN_SCORE = 0.01
SIMCLR_TOPN = 2                   # 검출 하나당 출력할 클래스 후보 수
SIMCLR_FUSE_ALPHA = 0.5           # 0=검출기 클래스만, 1=분류기 확률만. q = (1-a)*onehot(검출클래스) + a*분류확률
SIMCLR_TUNE_ALPHA = True          # True: val에서 SIMCLR_ALPHA_GRID를 비교해 alpha 자동 선택
SIMCLR_ALPHA_GRID = (0.25, 0.5, 0.75, 1.0)
SIMCLR_TTA = True                 # 0/90/180/270도 회전 평균
SIMCLR_REUSE_CACHE = True         # 같은 split·설정으로 학습한 가중치가 있으면 재사용


## 3. 데이터 경로


In [ ]:
DOWNLOAD_FROM_KAGGLE = True
KAGGLE_COMPETITION = "ai14-level-project"
LOCAL_DATA_ROOT = Path(r"D:\path\to\sprint_ai_project1_data")

if DOWNLOAD_FROM_KAGGLE:
    kagglehub.login()
    downloaded_path = Path(kagglehub.competition_download(KAGGLE_COMPETITION))
    nested_path = downloaded_path / "sprint_ai_project1_data"
    DATA_ROOT = nested_path if nested_path.exists() else downloaded_path
else:
    DATA_ROOT = LOCAL_DATA_ROOT.expanduser()

TRAIN_IMAGE_DIR = DATA_ROOT / "train_images"
TRAIN_ANNOTATION_DIR = DATA_ROOT / "train_annotations"
TEST_IMAGE_DIR = DATA_ROOT / "test_images"

for required_dir in (TRAIN_IMAGE_DIR, TRAIN_ANNOTATION_DIR, TEST_IMAGE_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(f"필수 폴더가 없습니다: {required_dir}")

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def list_images(directory: Path) -> list[Path]:
    return sorted(
        p for p in Path(directory).iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )


train_image_paths = list_images(TRAIN_IMAGE_DIR)
test_paths = list_images(TEST_IMAGE_DIR)
annotation_paths = sorted(TRAIN_ANNOTATION_DIR.rglob("*.json"))

print(f"DATA_ROOT       : {DATA_ROOT}")
print(f"Train 이미지 수: {len(train_image_paths):,}")
print(f"Annotation 수   : {len(annotation_paths):,}")
print(f"Test 이미지 수 : {len(test_paths):,}")

if EXPECTED_TEST_IMAGES is not None and len(test_paths) != EXPECTED_TEST_IMAGES:
    raise RuntimeError(
        f"Test 이미지 수 불일치: 예상={EXPECTED_TEST_IMAGES:,}, 실제={len(test_paths):,}"
    )


## 4. AI Hub JSON 보정 · partial annotation 병합


In [ ]:
image_stem_to_path = {p.stem: p for p in train_image_paths}
if len(image_stem_to_path) != len(train_image_paths):
    raise RuntimeError("서로 다른 확장자에 동일 stem을 가진 train 이미지가 있습니다.")

stem_to_annotation_paths = defaultdict(list)
for annotation_path in annotation_paths:
    stem_to_annotation_paths[annotation_path.stem].append(annotation_path)

valid_stems = sorted(set(image_stem_to_path) & set(stem_to_annotation_paths))
images_without_annotation = set(image_stem_to_path) - set(stem_to_annotation_paths)
annotations_without_image = set(stem_to_annotation_paths) - set(image_stem_to_path)

print(f"매핑 가능한 이미지         : {len(valid_stems):,}")
print(f"annotation 없는 이미지     : {len(images_without_annotation):,}")
print(f"이미지 없는 annotation stem: {len(annotations_without_image):,}")



def load_aihub_json(json_path: Path) -> dict:
    """AI Hub의 잘못된 `\"area\": ,`만 null로 최소 보정해 읽습니다."""
    text = Path(json_path).read_text(encoding="utf-8")
    text = re.sub(r'("area"\s*:\s*),', r'\1null,', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"JSON 파싱 실패: {json_path}\n"
            f"line={exc.lineno}, column={exc.colno}: {exc.msg}"
        ) from exc


def clip_xywh(bbox, image_w: int, image_h: int):
    try:
        x, y, bw, bh = map(float, bbox)
    except (TypeError, ValueError):
        return None
    if not np.isfinite([x, y, bw, bh]).all() or bw <= 0 or bh <= 0:
        return None
    x1 = min(max(x, 0.0), float(image_w))
    y1 = min(max(y, 0.0), float(image_h))
    x2 = min(max(x + bw, 0.0), float(image_w))
    y2 = min(max(y + bh, 0.0), float(image_h))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2 - x1, y2 - y1]


def build_image_records(stems, grouped_paths):
    image_records = {}
    code_to_name = {}
    stats = Counter()

    for stem in tqdm(stems, desc="AI Hub JSON 병합"):
        source_path = image_stem_to_path[stem]
        with Image.open(source_path) as image:
            actual_w, actual_h = image.size

        annotations = []
        seen_annotations = set()

        for annotation_path in grouped_paths[stem]:
            data = load_aihub_json(annotation_path)
            if not data.get("images"):
                raise ValueError(f"images 정보가 없는 annotation: {annotation_path}")

            image_meta = data["images"][0]
            drug_code = str(image_meta["dl_mapping_code"]).strip()
            drug_name = str(image_meta["dl_name"]).strip()
            meta_size = (int(image_meta["width"]), int(image_meta["height"]))
            if meta_size != (actual_w, actual_h):
                raise ValueError(
                    f"실제 이미지/JSON 크기 불일치: {source_path.name}, "
                    f"actual={(actual_w, actual_h)}, json={meta_size}"
                )

            old_name = code_to_name.get(drug_code)
            if old_name is not None and old_name != drug_name:
                raise ValueError(
                    f"같은 dl_mapping_code의 이름 불일치: {drug_code} -> "
                    f"{old_name!r} / {drug_name!r}"
                )

            for ann in data.get("annotations", []):
                bbox = ann.get("bbox", [])
                if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
                    stats["invalid_bbox"] += 1
                    continue

                clipped = clip_xywh(bbox, actual_w, actual_h)
                if clipped is None:
                    stats["invalid_bbox"] += 1
                    continue

                x, y, bw, bh = clipped
                dedup_key = (
                    drug_code,
                    round(x, 4), round(y, 4), round(bw, 4), round(bh, 4),
                )
                if dedup_key in seen_annotations:
                    stats["duplicate_bbox"] += 1
                    continue
                seen_annotations.add(dedup_key)

                raw_area = ann.get("area")
                try:
                    raw_area = float(raw_area)
                    if not math.isfinite(raw_area) or raw_area <= 0:
                        raise ValueError
                except (TypeError, ValueError):
                    stats["repaired_area"] += 1

                annotations.append({
                    "bbox": [x, y, bw, bh],
                    "category_code": drug_code,
                    "area": bw * bh,  # clip 이후 bbox와 항상 일치
                    "iscrowd": 0,
                })
                code_to_name[drug_code] = drug_name

        if not annotations:
            stats["empty_images"] += 1
            continue

        image_records[stem] = {
            "image_path": str(source_path),
            "file_name": source_path.name,
            "width": actual_w,
            "height": actual_h,
            "annotations": annotations,
        }

    return image_records, code_to_name, stats


image_records, code_to_name, merge_stats = build_image_records(
    valid_stems, stem_to_annotation_paths
)

print(f"최종 image record: {len(image_records):,}")
print(f"실제 약품 클래스 : {len(code_to_name):,}")
print(f"병합 통계        : {dict(merge_stats)}")


In [ ]:
# 4-1. 동일 이미지 · 동일 bbox · 서로 다른 클래스 충돌 검사 및 제외
# 원본 JSON은 수정하지 않고, 학습용 image_records에서만 제외합니다.

BBOX_KEY_DECIMALS = 4
EXCLUDE_CLASS_CONFLICT_IMAGES = True

ANNOTATION_AUDIT_DIR = PROJECT_DIR / f"{RUN_NAME}_audit"   # 학습 폴더(OUTPUT_DIR)를 비워 두기 위해 분리
ANNOTATION_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CONFLICT_COLUMNS = [
    "stem",
    "image_file",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h",
    "category_code",
    "all_conflicting_category_codes",
    "merged_annotation_index",
    "source_json_files",
]

def bbox_geometry_key(bbox, decimals=BBOX_KEY_DECIMALS):
    """클립·보정된 COCO XYWH bbox를 클래스와 무관하게 비교하기 위한 키."""
    x, y, w, h = map(float, bbox)
    return tuple(round(v, decimals) for v in (x, y, w, h))


conflict_rows = []
class_conflict_stems = set()

for stem, record in tqdm(image_records.items(), desc="클래스 충돌 annotation 검사"):
    # bbox 좌표만 기준으로 묶음: 클래스는 의도적으로 키에 넣지 않음
    geometry_to_annotations = defaultdict(list)

    for ann_idx, ann in enumerate(record["annotations"]):
        geometry_to_annotations[bbox_geometry_key(ann["bbox"])].append(
            (ann_idx, ann)
        )

    for (x, y, w, h), grouped_annotations in geometry_to_annotations.items():
        category_codes = sorted({
            str(ann["category_code"])
            for _, ann in grouped_annotations
        })

        # 같은 bbox에 클래스가 2개 이상이면 충돌
        if len(category_codes) < 2:
            continue

        class_conflict_stems.add(stem)
        source_json_files = " | ".join(
            str(path) for path in stem_to_annotation_paths[stem]
        )

        for ann_idx, ann in grouped_annotations:
            conflict_rows.append({
                "stem": stem,
                "image_file": record["file_name"],
                "bbox_x": x,
                "bbox_y": y,
                "bbox_w": w,
                "bbox_h": h,
                "category_code": ann["category_code"],
                "all_conflicting_category_codes": " | ".join(category_codes),
                "merged_annotation_index": ann_idx,
                "source_json_files": source_json_files,
            })

class_conflict_df = pd.DataFrame(conflict_rows, columns=CONFLICT_COLUMNS)
conflict_csv_path = ANNOTATION_AUDIT_DIR / "same_bbox_different_class_conflicts.csv"
class_conflict_df.to_csv(conflict_csv_path, index=False, encoding="utf-8-sig")

print(f"동일 bbox · 다른 클래스 충돌 이미지 수: {len(class_conflict_stems):,}")
print(f"충돌 annotation 행 수              : {len(class_conflict_df):,}")
print(f"상세 보고서                         : {conflict_csv_path.resolve()}")

if not class_conflict_df.empty:
    display(class_conflict_df.head(20))

if EXCLUDE_CLASS_CONFLICT_IMAGES and class_conflict_stems:
    image_count_before = len(image_records)
    box_count_before = sum(
        len(record["annotations"]) for record in image_records.values()
    )

    # 원본 image_records는 이 셀을 다시 실행하면 Cell 12에서 재구성됨.
    # 별도 변수에도 보관해 두어 즉시 확인 가능하게 함.
    excluded_class_conflict_records = {
        stem: image_records[stem]
        for stem in class_conflict_stems
    }

    image_records = {
        stem: record
        for stem, record in image_records.items()
        if stem not in class_conflict_stems
    }

    remaining_codes = {
        ann["category_code"]
        for record in image_records.values()
        for ann in record["annotations"]
    }
    disappeared_codes = sorted(set(code_to_name) - remaining_codes)

    if disappeared_codes:
        raise RuntimeError(
            "충돌 이미지 제외 후 학습 데이터에서 완전히 사라진 클래스가 있습니다. "
            "이 경우 해당 이미지를 무조건 제외하지 말고 CSV를 보고 직접 판정해야 합니다.\n"
            f"사라진 클래스: {disappeared_codes}"
        )

    box_count_after = sum(
        len(record["annotations"]) for record in image_records.values()
    )

    print("\n충돌 이미지 제외 완료")
    print(f"이미지: {image_count_before:,} → {len(image_records):,}")
    print(f"bbox  : {box_count_before:,} → {box_count_after:,}")
else:
    print("\n제외할 클래스 충돌 이미지는 없습니다.")


## 5. 0-based 모델 클래스 · 제출 category 매핑


In [ ]:
sorted_codes = sorted(code_to_name)
code_to_model_id = {code: idx for idx, code in enumerate(sorted_codes)}
model_id_to_code = {idx: code for code, idx in code_to_model_id.items()}
model_id_to_name = {idx: code_to_name[code] for idx, code in model_id_to_code.items()}


def dl_code_to_submission_id(class_code: str) -> int:
    match = re.fullmatch(r"K-(\d+)", str(class_code).strip())
    if match is None:
        raise ValueError(f"예상하지 못한 dl_mapping_code: {class_code!r}")
    return int(match.group(1))


code_to_submission_id = {
    code: dl_code_to_submission_id(code) for code in sorted_codes
}

num_classes = len(sorted_codes)
if num_classes != EXPECTED_NUM_CLASSES:
    raise RuntimeError(
        f"클래스 수 불일치: 예상={EXPECTED_NUM_CLASSES}, 실제={num_classes}. "
        "annotation 폴더와 병합 결과를 확인하세요."
    )

assert set(model_id_to_code) == set(range(num_classes))
assert len(set(code_to_submission_id.values())) == num_classes

mapping_preview = pd.DataFrame([
    {
        "model_id": model_id,
        "dl_mapping_code": model_id_to_code[model_id],
        "submission_category_id": code_to_submission_id[model_id_to_code[model_id]],
        "name": model_id_to_name[model_id],
    }
    for model_id in range(num_classes)
])
display(mapping_preview.head(10))


# 6. Train / Valid 분할 (augmentation family 단위)

색상/채도 변형은 bbox가 같고, H/V flip은 bbox 중심이 `(x, y) → (1-x, 1-y)`로 바뀝니다.
flip 불변 signature로 파생본 family를 찾고, **family 전체를 하나의 분할에만** 넣어 train/valid 누수를 막습니다.


In [ ]:
def augmentation_family_signature(record: dict, decimals: int = 3) -> str:
    width = float(record["width"])
    height = float(record["height"])
    tokens = []
    for ann in record["annotations"]:
        x, y, bw, bh = ann["bbox"]
        cx = (x + bw / 2) / width
        cy = (y + bh / 2) / height
        token = (
            ann["category_code"],
            round(min(cx, 1.0 - cx), decimals),
            round(min(cy, 1.0 - cy), decimals),
            round(bw / width, decimals),
            round(bh / height, decimals),
        )
        tokens.append(token)
    payload = json.dumps(sorted(tokens), ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


def family_warn_or_raise(message: str) -> None:
    """FAMILY_STRICT=True면 중단, False면 경고만 출력하고 계속 진행합니다."""
    if FAMILY_STRICT:
        raise RuntimeError(message)
    print("⚠ 경고:", message)


family_to_stems = defaultdict(list)
for stem, record in image_records.items():
    signature = augmentation_family_signature(record, GROUP_GEOMETRY_DECIMALS)
    family_to_stems[signature].append(stem)

group_sizes = Counter(len(stems) for stems in family_to_stems.values())
singleton_ratio = group_sizes.get(1, 0) / len(family_to_stems)
largest_group = max(map(len, family_to_stems.values()))

print(f"family 수          : {len(family_to_stems):,}")
print(f"family 크기 분포   : {dict(sorted(group_sizes.items()))}")
print(f"singleton family 비율: {singleton_ratio:.1%}")
print(f"최대 family 크기   : {largest_group}")

if singleton_ratio > 0.50:
    family_warn_or_raise(
        "자동 family 탐지 결과 singleton이 50%를 넘습니다. 파생본이 거의 탐지되지 않았거나 데이터에 파생본이 적다는 뜻입니다. "
        "파일명 규칙에 원본 ID가 있다면 augmentation_family_signature()를 그 ID 추출 방식으로 바꾸세요."
    )
if largest_group > 8:
    family_warn_or_raise(
        f"family 크기 {largest_group}이 비정상적으로 큽니다. 서로 다른 원본이 같은 geometry signature로 충돌했는지 확인하세요."
    )


In [ ]:
family_ids = sorted(family_to_stems)
family_labels = []
family_weights = []

for family_id in family_ids:
    stems = family_to_stems[family_id]
    present_codes = {
        ann["category_code"]
        for stem in stems
        for ann in image_records[stem]["annotations"]
    }
    row = np.zeros(num_classes, dtype=np.uint8)
    for code in present_codes:
        row[code_to_model_id[code]] = 1
    family_labels.append(row)
    family_weights.append(len(stems))

family_labels = np.asarray(family_labels)
family_weights = np.asarray(family_weights)
overall_class_images = np.zeros(num_classes, dtype=np.int64)
for record in image_records.values():
    for code in {ann["category_code"] for ann in record["annotations"]}:
        overall_class_images[code_to_model_id[code]] += 1

# 하나의 family에만 존재하는 클래스는 그 family를 반드시 train에 둡니다.
class_family_support = family_labels.sum(axis=0)
forced_train_family_idx = set(
    np.flatnonzero(family_labels[:, class_family_support == 1].any(axis=1)).tolist()
)
free_family_idx = np.asarray(
    [i for i in range(len(family_ids)) if i not in forced_train_family_idx],
    dtype=int,
)
desired_valid_images = round(len(image_records) * VAL_RATIO)
free_image_count = int(family_weights[free_family_idx].sum())
free_valid_ratio = desired_valid_images / free_image_count
if not 0 < free_valid_ratio < 1:
    raise RuntimeError(
        f"강제 train family 제외 후 valid 비율을 만들 수 없습니다: {free_valid_ratio:.3f}"
    )

print(f"희소 클래스 때문에 train 고정된 family: {len(forced_train_family_idx)}")


def split_score(train_family_idx, valid_family_idx):
    train_stems_ = [
        stem for idx in train_family_idx for stem in family_to_stems[family_ids[idx]]
    ]
    valid_stems_ = [
        stem for idx in valid_family_idx for stem in family_to_stems[family_ids[idx]]
    ]

    train_counts = np.zeros(num_classes, dtype=np.int64)
    valid_counts = np.zeros(num_classes, dtype=np.int64)
    for stem in train_stems_:
        for code in {a["category_code"] for a in image_records[stem]["annotations"]}:
            train_counts[code_to_model_id[code]] += 1
    for stem in valid_stems_:
        for code in {a["category_code"] for a in image_records[stem]["annotations"]}:
            valid_counts[code_to_model_id[code]] += 1

    missing_train = int((train_counts == 0).sum())
    missing_valid = int((valid_counts == 0).sum())
    ratio_error = abs(len(valid_stems_) / len(image_records) - VAL_RATIO)
    valid_fraction = np.divide(
        valid_counts,
        overall_class_images,
        out=np.zeros_like(valid_counts, dtype=float),
        where=overall_class_images > 0,
    )
    strat_error = float(np.mean(np.abs(valid_fraction - VAL_RATIO)))
    # train class 누락은 사실상 허용하지 않고, valid 누락도 강한 penalty를 줍니다.
    # 사전식 비교: train 누락 0개가 어떤 비율 개선보다 항상 우선합니다.
    score = (missing_train, missing_valid, ratio_error * 10 + strat_error)
    return score, train_stems_, valid_stems_, train_counts, valid_counts


best = None
for candidate_seed in range(SEED, SEED + N_SPLIT_CANDIDATES):
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=free_valid_ratio,
        random_state=candidate_seed,
    )
    free_train_local, free_valid_local = next(
        splitter.split(
            np.zeros((len(free_family_idx), 1)),
            family_labels[free_family_idx],
        )
    )
    train_idx = np.asarray(sorted(
        forced_train_family_idx | set(free_family_idx[free_train_local].tolist())
    ))
    valid_idx = free_family_idx[free_valid_local]
    candidate = split_score(train_idx, valid_idx)
    if best is None or candidate[0] < best[0]:
        best = (*candidate, candidate_seed, train_idx, valid_idx)

(
    best_score,
    train_stems,
    valid_stems,
    train_class_counts,
    valid_class_counts,
    selected_seed,
    train_family_idx,
    valid_family_idx,
) = best

train_stems = sorted(train_stems)
valid_stems = sorted(valid_stems)
train_families = {family_ids[i] for i in train_family_idx}
valid_families = {family_ids[i] for i in valid_family_idx}

assert not (train_families & valid_families), "family leakage가 발견되었습니다."
assert not (set(train_stems) & set(valid_stems))
assert len(train_stems) + len(valid_stems) == len(image_records)
if (train_class_counts == 0).any():
    raise RuntimeError("Train에 빠진 클래스가 있습니다. split 알고리즘을 점검하세요.")

print(f"선택 seed  : {selected_seed}")
print(f"Train      : {len(train_stems):,}장 / {len(train_families):,} families")
print(f"Valid      : {len(valid_stems):,}장 / {len(valid_families):,} families")
print(f"Valid 비율 : {len(valid_stems) / len(image_records):.2%}")
print(f"Valid 누락 클래스: {int((valid_class_counts == 0).sum())}")

if (valid_class_counts == 0).any():
    missing_ids = np.flatnonzero(valid_class_counts == 0).tolist()
    print("경고 - valid 누락 클래스:", [model_id_to_code[i] for i in missing_ids])

split_stats = mapping_preview.copy()
split_stats["all_images"] = overall_class_images
split_stats["train_images"] = train_class_counts
split_stats["valid_images"] = valid_class_counts
display(split_stats.sort_values("all_images").head(20))


## 7. YOLO dataset 생성

```text
images/train, images/val
labels/train, labels/val
data.yaml
```

라벨 한 줄은 `class x_center y_center width height`이며 모두 0~1 정규화 값입니다.


In [ ]:
YOLO_ROOT = DATA_ROOT / f"yolo_family_seed{selected_seed}_val{int(VAL_RATIO * 100)}"
REBUILD_YOLO_DATASET = True

def hardlink_or_copy(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists(): return
    try: os.link(source, destination)
    except OSError: shutil.copy2(source, destination)

if REBUILD_YOLO_DATASET and YOLO_ROOT.exists():
    if not YOLO_ROOT.name.startswith("yolo_family_"):
        raise RuntimeError(f"안전하지 않은 삭제 대상: {YOLO_ROOT}")
    shutil.rmtree(YOLO_ROOT)

def export_yolo_split(split_name: str, stems: list[str]) -> None:
    image_dir, label_dir = YOLO_ROOT / "images" / split_name, YOLO_ROOT / "labels" / split_name
    image_dir.mkdir(parents=True, exist_ok=True); label_dir.mkdir(parents=True, exist_ok=True)
    for stem in tqdm(stems, desc=f"YOLO {split_name}"):
        record = image_records[stem]
        hardlink_or_copy(Path(record["image_path"]), image_dir / record["file_name"])
        lines = []
        for ann in record["annotations"]:
            x, y, bw, bh = map(float, ann["bbox"])
            xc, yc = (x + bw / 2) / record["width"], (y + bh / 2) / record["height"]
            nw, nh = bw / record["width"], bh / record["height"]
            if not (0 < nw <= 1 and 0 < nh <= 1 and 0 <= xc <= 1 and 0 <= yc <= 1):
                raise ValueError(f"YOLO 정규화 bbox 오류: {stem} / {ann}")
            lines.append(f"{code_to_model_id[ann['category_code']]} {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}")
        (label_dir / f"{Path(record['file_name']).stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

export_yolo_split("train", train_stems)
export_yolo_split("val", valid_stems)

DATA_YAML = YOLO_ROOT / "data.yaml"
data_config = {
    "path": str(YOLO_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": num_classes,
    "names": {i: f"{model_id_to_code[i]} | {model_id_to_name[i]}" for i in range(num_classes)},
}
DATA_YAML.write_text(yaml.safe_dump(data_config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(f"YOLO_ROOT: {YOLO_ROOT}")
print(f"train={len(train_stems):,}, val={len(valid_stems):,}, classes={num_classes}")
print(DATA_YAML.read_text(encoding="utf-8")[:500])


## 8. YOLO dataset 무결성 검사 · label 시각화


In [ ]:
def validate_yolo_labels(split_name: str, expected_stems: list[str]):
    image_dir, label_dir = YOLO_ROOT / "images" / split_name, YOLO_ROOT / "labels" / split_name
    images = list_images(image_dir)
    labels = sorted(label_dir.glob("*.txt"))
    assert len(images) == len(expected_stems) == len(labels), (len(images), len(labels), len(expected_stems))
    class_ids = []
    for path in labels:
        rows = [line.split() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not rows: raise RuntimeError(f"빈 label 파일: {path}")
        for row in rows:
            if len(row) != 5: raise RuntimeError(f"YOLO 열 수 오류: {path}: {row}")
            cls, *values = row
            class_ids.append(int(cls)); values = [float(v) for v in values]
            if not (0 <= int(cls) < num_classes and all(0 <= v <= 1 for v in values)):
                raise RuntimeError(f"YOLO 범위 오류: {path}: {row}")
    print(f"{split_name:>5}: images={len(images):,}, labels={len(labels):,}, boxes={len(class_ids):,}")

validate_yolo_labels("train", train_stems)
validate_yolo_labels("val", valid_stems)

def show_ground_truth(stem, figsize=(10, 8)):
    record = image_records[stem]
    image = np.asarray(Image.open(record["image_path"]).convert("RGB")).copy()
    xyxy = np.array([[x, y, x + w, y + h] for x,y,w,h in (a["bbox"] for a in record["annotations"])], dtype=float)
    class_id = np.array([code_to_model_id[a["category_code"]] for a in record["annotations"]], dtype=int)
    det = sv.Detections(xyxy=xyxy, class_id=class_id)
    labels = [f"{model_id_to_code[c]}" for c in class_id]
    canvas = sv.BoxAnnotator(thickness=3).annotate(image, det)
    canvas = sv.LabelAnnotator(text_scale=.45).annotate(canvas, det, labels)
    plt.figure(figsize=figsize); plt.imshow(canvas); plt.title(f"GT · {stem}"); plt.axis("off"); plt.show()

for sample_stem in random.sample(valid_stems, min(4, len(valid_stems))): show_ground_truth(sample_stem)


## 9. 50 iteration 속도 시험 (선택)

`RUN_SPEED_TEST=True`일 때 약 50 training iteration만 수행하고 validation은 생략합니다.


In [ ]:
if RUN_SPEED_TEST:
    speed_fraction = min(1.0, (50 * BATCH) / len(train_stems))
    speed_model = YOLO(MODEL_WEIGHTS)
    speed_start = time.perf_counter()
    speed_model.train(
        data=str(DATA_YAML), epochs=1, fraction=speed_fraction, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
        device=0, workers=NUM_WORKERS, amp=AMP, deterministic=DETERMINISTIC, val=False, cache=False,
        project=str(PROJECT_DIR), name=f"{RUN_NAME}_speed50", exist_ok=False,
        save=False, plots=False, verbose=False,
        optimizer="auto", seed=SEED,
        degrees=0.0, translate=0.05, scale=0.10, shear=0.0, perspective=0.0,
        flipud=0.0, fliplr=0.5, mosaic=0.0, mixup=0.0,
    )
    speed_seconds = time.perf_counter() - speed_start
    estimated_train_batches = math.ceil(len(train_stems) / BATCH)
    # 초기 모델 준비 시간을 제외하지 못하므로 보수적으로 validation 오버헤드 20%를 더합니다.
    seconds_per_iteration = speed_seconds / 50
    estimated_total_hours = seconds_per_iteration * estimated_train_batches * EPOCHS * 1.20 / 3600
    print(f"50 iteration wall time: {speed_seconds:.1f}초")
    print(f"추정 학습 batch/epoch: {estimated_train_batches:,}")
    print(f"30 epoch 보수적 예상: {estimated_total_hours:.2f}시간 (validation 20% 포함)")
    if estimated_total_hours > 5:
        print("5시간 초과 예상 → yolo26n.pt 또는 IMGSZ=448로 속도 시험을 다시 하세요.")
    else:
        print("5시간 목표 통과 → RUN_SPEED_TEST=False, RUN_TRAINING=True로 바꿔 학습하세요.")
    del speed_model
    gc.collect(); torch.cuda.empty_cache()
else:
    print("RUN_SPEED_TEST=False: 속도 시험 생략")


## 10. Epoch별 Loss · 대회 mAP 출력 설정

기본 validation이 계산한 클래스별 AP 배열(0.50, 0.55, …, 0.95)에서 0.75~0.95 다섯 칸만 평균합니다. 따라서 매 epoch마다 validation 추론을 추가로 한 번 더 하지 않습니다.

표시 형식: `Epoch 01/30 | Loss: 12.34567 | mAP@[0.75:0.95]: 0.12345`


In [ ]:
train_loss_hist = []
val_map_hist = []
val_epoch_hist = []
BEST_COMPETITION_MAP = -float("inf")
EPOCH_SUMMARY_PATH = OUTPUT_DIR / "epoch_summary.csv"

def _as_float(value):
    return float(value.detach().cpu()) if hasattr(value, "detach") else float(value)

def _competition_map_from_validator(trainer):
    # Ultralytics YOLO detection validator: IoU axis = [.50, .55, ..., .95].
    all_ap = np.asarray(trainer.validator.metrics.box.all_ap, dtype=float)
    if all_ap.ndim != 2 or all_ap.shape[1] < 10:
        raise RuntimeError(f"예상하지 못한 Ultralytics AP 배열 형태: {all_ap.shape}")
    return float(np.nanmean(all_ap[:, 5:10]))  # .75, .80, .85, .90, .95

def report_competition_epoch(trainer):
    """기본 validation 직후 1회 실행: 화면에는 요청한 두 지표만 출력."""
    global BEST_COMPETITION_MAP, EPOCH_SUMMARY_PATH

    if trainer.tloss is None:
        return
    loss_values = trainer.tloss.values() if isinstance(trainer.tloss, dict) else trainer.tloss
    train_loss = sum(_as_float(value) for value in loss_values)
    competition_map = _competition_map_from_validator(trainer)
    epoch_number = int(trainer.epoch) + 1

    train_loss_hist.append(train_loss)
    val_map_hist.append(competition_map)
    val_epoch_hist.append(epoch_number)

    # Ultralytics가 실제 생성한 현재 실행 폴더를 기준으로 저장
    EPOCH_SUMMARY_PATH = Path(trainer.save_dir) / "epoch_summary.csv"
    EPOCH_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

    pd.DataFrame({
        "epoch": val_epoch_hist,
        "train_loss": train_loss_hist,
        "map_75_95": val_map_hist,
    }).to_csv(EPOCH_SUMMARY_PATH, index=False)

    if USE_WANDB and run is not None:
        learning_rate = trainer.optimizer.param_groups[0]["lr"]

        wandb.log(
            {
                "epoch": epoch_number,
                "train/loss": train_loss,
                "validation/mAP_75_95": competition_map,
                "learning_rate": learning_rate,
            },
            step=epoch_number,
        )

    # trainer.last는 이 callback보다 앞서 현재 epoch EMA checkpoint로 저장되어 있습니다.
    if competition_map > BEST_COMPETITION_MAP:
        BEST_COMPETITION_MAP = competition_map
        shutil.copy2(trainer.last, trainer.wdir / "best_competition_map.pt")

    print(
        f"Epoch {epoch_number:02d}/{trainer.epochs} | "
        f"Loss: {train_loss:.5f} | "
        f"mAP@[0.75:0.95]: {competition_map:.5f}"
    )

class _SilentTQDM:
    """Ultralytics batch progress bar를 감춰 epoch 요약만 남깁니다."""
    def __init__(self, iterable, *args, **kwargs):
        self.iterable = iterable
    def __iter__(self):
        return iter(self.iterable)
    def set_description(self, *args, **kwargs):
        pass
    def close(self):
        pass

def configure_quiet_epoch_log():
    if QUIET_EPOCH_LOG:
        # batch loss/progress/기본 COCO mAP 출력을 감추고 위 callback만 보입니다.
        from ultralytics.utils import LOGGER
        import ultralytics.engine.trainer as ultralytics_trainer
        LOGGER.setLevel(logging.ERROR)
        ultralytics_trainer.TQDM = _SilentTQDM


## 11. 학습

YOLO26s COCO pretrained weight를 118개 클래스에 fine-tuning합니다.

- `optimizer="auto"`로 YOLO26의 권장 optimizer를 사용합니다.
- synthetic family를 train/val에 분리했으므로 mosaic·mixup은 끕니다.
- 기본 validation은 매 epoch 한 번만 수행하고, 출력은 아래 callback의 `Loss`와 `mAP@[0.75:0.95]`만 남깁니다.


In [ ]:
#@title W&B 실험 시작
run = None

if USE_WANDB:
    try:
        wandb.login()          # 계정이 있으니 2 입력 후 API Key 입력 (이미 로그인돼 있으면 바로 통과)
        run = wandb.init(
            entity=WANDB_ENTITY,
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            job_type="train",
            config={
                "model": MODEL_WEIGHTS,
                "epochs": EPOCHS,
                "imgsz": IMGSZ,
                "batch_size": BATCH,
                "nbs": NBS,
                "num_classes": EXPECTED_NUM_CLASSES,
                "val_ratio": VAL_RATIO,
                "seed": SEED,
                "amp": AMP,
                "workers": NUM_WORKERS,
                "competition_ious": COMPETITION_IOUS,
                "use_sam": USE_SAM,
                "use_simclr": USE_SIMCLR,
            },
        )
    except Exception as exc:
        print(f"⚠ W&B 시작 실패 → 로컬 기록으로 진행합니다: {type(exc).__name__}: {str(exc)[:150]}")
        run = None
        USE_WANDB = False


In [ ]:
if RUN_TRAINING:
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):  # (audit 폴더는 별도 경로로 분리됨)
        raise RuntimeError(f"기존 실행 폴더가 있습니다: {OUTPUT_DIR}. 새 실험이면 RUN_NAME을 바꾸세요.")

    model = YOLO(MODEL_WEIGHTS)
    model.add_callback("on_fit_epoch_end", report_competition_epoch)

    train_start = time.perf_counter()
    train_results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
        device=0, workers=NUM_WORKERS, amp=AMP, deterministic=DETERMINISTIC,
        optimizer="auto", cos_lr=True, warmup_epochs=1.0,
        project=str(PROJECT_DIR), name=RUN_NAME, exist_ok=False,
        pretrained=True, cache=False, patience=0, save=True, save_period=5, val=True,
        plots=False, verbose=True, seed=SEED,
        degrees=0.0, translate=0.05, scale=0.10, shear=0.0, perspective=0.0,
        flipud=0.0, fliplr=0.5, mosaic=0.0, mixup=0.0, erasing=0.0,
        close_mosaic=0,
    )
    print(f"학습 소요 시간: {(time.perf_counter() - train_start)/3600:.2f}시간")
elif RESUME:
    last_pt = OUTPUT_DIR / "weights" / "last.pt"
    if not last_pt.exists():
        raise FileNotFoundError(last_pt)
    model = YOLO(str(last_pt))
    model.add_callback("on_fit_epoch_end", report_competition_epoch)
    train_results = model.train(resume=True)
else:
    print("학습 생략: 기존 best.pt를 다음 셀에서 사용합니다.")


## 12. best checkpoint 로드 · 기본 COCO 평가


In [ ]:
_trainer = getattr(globals().get("model", None), "trainer", None)
if _trainer is not None and getattr(_trainer, "save_dir", None):
    RUN_OUTPUT_DIR = Path(_trainer.save_dir)                # 이번 세션에서 학습한 실행 폴더
elif EXISTING_RUN_DIR:
    RUN_OUTPUT_DIR = Path(EXISTING_RUN_DIR)
else:
    _found = sorted(PROJECT_DIR.glob(f"{RUN_NAME}*/weights/best.pt"), key=lambda p: p.stat().st_mtime)
    if not _found:
        raise FileNotFoundError(
            f"기존 best.pt를 찾지 못했습니다: {PROJECT_DIR / (RUN_NAME + '*')}. EXISTING_RUN_DIR를 지정하세요."
        )
    RUN_OUTPUT_DIR = _found[-1].parent.parent
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)

BEST_CHECKPOINT = RUN_OUTPUT_DIR / "weights" / "best_competition_map.pt"
if not BEST_CHECKPOINT.exists():
    BEST_CHECKPOINT = RUN_OUTPUT_DIR / "weights" / "best.pt"

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(BEST_CHECKPOINT)

gc.collect()
torch.cuda.empty_cache()

best_model = YOLO(str(BEST_CHECKPOINT))

official_eval = best_model.val(
    data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, device=0, workers=NUM_WORKERS,
    amp=AMP, split="val", plots=True, max_det=EVAL_MAX_DETECTIONS, verbose=True,
)
print(f"COCO mAP@[.50:.95]: {official_eval.box.map:.5f}")
print(f"COCO mAP@.50      : {official_eval.box.map50:.5f}")


In [ ]:
#@title 최고 모델 및 최종 평가 결과를 W&B에 기록

if USE_WANDB and run is not None:

    # 최종 best model 평가 결과 기록
    wandb.log({
        "final/coco_mAP_50_95": float(official_eval.box.map),
        "final/coco_mAP_50": float(official_eval.box.map50),
        "final/best_competition_mAP_75_95": float(BEST_COMPETITION_MAP),
    })

    # Best model Artifact
    model_artifact = wandb.Artifact(
        name=f"{MODEL_TAG}-{run.id}",
        type="model",
        description=f"Best checkpoint from {WANDB_RUN_NAME}",
        metadata={
            "model": MODEL_WEIGHTS,
            "epochs": EPOCHS,
            "imgsz": IMGSZ,
            "batch_size": BATCH,
            "nbs": NBS,
            "num_classes": EXPECTED_NUM_CLASSES,
            "val_ratio": VAL_RATIO,
            "seed": SEED,
            "amp": AMP,
            "best_map_75_95": BEST_COMPETITION_MAP,
        }
    )

    model_artifact.add_file(str(BEST_CHECKPOINT))

    run.log_artifact(
        model_artifact,
        aliases=["best"]
    )

    run.finish()

    print(f"{MODEL_TAG} 실험 및 Artifact 업로드 완료")


## 12. 대회 지표 mAP@[0.75:0.95] 직접 계산 (base) · 추가 기능 공통 유틸


In [ ]:
# ══════════════════════════════════════════════════════════════
# 추가 기능 공통 유틸 · 검출 어댑터 · 평가 함수 (v2)
# ══════════════════════════════════════════════════════════════
import contextlib

RUN_OUTPUT_DIR = Path(RUN_OUTPUT_DIR) if "RUN_OUTPUT_DIR" in globals() else Path(OUTPUT_DIR)
ADDON_DIR = RUN_OUTPUT_DIR / "addons"
ADDON_DIR.mkdir(parents=True, exist_ok=True)


def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def make_pred(xyxy, scores, labels, wh, max_det=None):
    """검출 결과를 점수 내림차순 dict로 통일: xyxy(N,4) float32, scores(N,), labels(N,), wh=(W,H)."""
    xyxy = np.asarray(xyxy, dtype=np.float32).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    labels = np.asarray(labels).astype(np.int64).reshape(-1)
    order = np.argsort(-scores, kind="stable")
    if max_det is not None:
        order = order[:max_det]
    return {
        "xyxy": xyxy[order],
        "scores": scores[order],
        "labels": labels[order],
        "wh": (int(wh[0]), int(wh[1])),
    }


def clip_xyxy(xyxy, wh):
    out = np.asarray(xyxy, dtype=np.float32).reshape(-1, 4).copy()
    out[:, [0, 2]] = np.clip(out[:, [0, 2]], 0, wh[0])
    out[:, [1, 3]] = np.clip(out[:, [1, 3]], 0, wh[1])
    return out


def box_iou_pairwise(a, b):
    """같은 행끼리의 IoU. a, b: (N,4) xyxy."""
    a = np.asarray(a, dtype=np.float32).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float32).reshape(-1, 4)
    ix1, iy1 = np.maximum(a[:, 0], b[:, 0]), np.maximum(a[:, 1], b[:, 1])
    ix2, iy2 = np.minimum(a[:, 2], b[:, 2]), np.minimum(a[:, 3], b[:, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.maximum(area_a + area_b - inter, 1e-6)


def detect_batch(paths, max_det=None, conf=None):
    """[이미지 경로] → [pred dict]. (YOLO 어댑터)"""
    max_det = EVAL_MAX_DETECTIONS if max_det is None else max_det
    conf = PRED_CONFIDENCE if conf is None else conf
    results = best_model.predict(
        [str(p) for p in paths], imgsz=IMGSZ, conf=conf, max_det=max_det,
        device=0, verbose=False,
    )
    if len(results) != len(paths):
        raise RuntimeError(f"예측 batch 길이 불일치: inputs={len(paths)}, outputs={len(results)}")
    out = []
    for result in results:
        height, width = result.orig_shape[:2]
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            out.append(make_pred(np.zeros((0, 4)), np.zeros(0), np.zeros(0), (width, height), max_det))
            continue
        out.append(make_pred(
            boxes.xyxy.detach().cpu().numpy(),
            boxes.conf.detach().cpu().numpy(),
            boxes.cls.detach().cpu().numpy(),
            (width, height), max_det,
        ))
    return out


def build_targets(stems):
    targets = {}
    for stem in stems:
        anns = image_records[stem]["annotations"]
        boxes = np.array(
            [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
            dtype=np.float32,
        ).reshape(-1, 4)
        labels = np.array([code_to_model_id[a["category_code"]] for a in anns], dtype=np.int64)
        targets[stem] = {"boxes": torch.as_tensor(boxes), "labels": torch.as_tensor(labels)}
    return targets


def _metric_dict(pred, n):
    return {
        "boxes": torch.as_tensor(pred["xyxy"][:n], dtype=torch.float32).reshape(-1, 4),
        "scores": torch.as_tensor(pred["scores"][:n], dtype=torch.float32),
        "labels": torch.as_tensor(pred["labels"][:n], dtype=torch.int64),
    }


def evaluate_variant(pred_map, stems, targets, chunk=64):
    """대회 지표 mAP@[.75:.95]. map20=이미지당 최대 20개, map8=제출과 같은 최대 8개."""
    metric20 = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", iou_thresholds=COMPETITION_IOUS,
        max_detection_thresholds=[1, 10, EVAL_MAX_DETECTIONS],
        class_metrics=False, backend="faster_coco_eval",
    )
    metric8 = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", iou_thresholds=COMPETITION_IOUS,
        max_detection_thresholds=[1, 5, SUBMISSION_MAX_DETECTIONS],
        class_metrics=False, backend="faster_coco_eval",
    )
    for stem_chunk in batched(list(stems), chunk):
        preds20 = [_metric_dict(pred_map[s], EVAL_MAX_DETECTIONS) for s in stem_chunk]
        preds8 = [_metric_dict(pred_map[s], SUBMISSION_MAX_DETECTIONS) for s in stem_chunk]
        tgts = [targets[s] for s in stem_chunk]
        metric20.update(preds20, tgts)
        metric8.update(preds8, tgts)
    return {
        "map20": float(metric20.compute()["map"]),
        "map8": float(metric8.compute()["map"]),
    }


In [ ]:
# ══ 검증셋 검출 캐시 + 기본(base) 대회 지표 ══
# 검출기를 val에 한 번만 돌려 두고, SAM / SimCLR 변형은 이 캐시에서 계산합니다.
val_paths = {stem: Path(image_records[stem]["image_path"]) for stem in valid_stems}
val_targets = build_targets(valid_stems)

val_raw = {}
for stem_batch in tqdm(list(batched(valid_stems, PRED_BATCH_SIZE)), desc="Validation 검출"):
    batch_preds = detect_batch([val_paths[s] for s in stem_batch], max_det=EVAL_MAX_DETECTIONS)
    for stem, pred in zip(stem_batch, batch_preds):
        val_raw[stem] = pred

variant_scores = {"base": evaluate_variant(val_raw, valid_stems, val_targets)}
competition_map = variant_scores["base"]["map20"]
submission_cap_map = variant_scores["base"]["map8"]
print(f"[base] Competition mAP@[.75:.95] (max {EVAL_MAX_DETECTIONS}): {competition_map:.5f}")
print(f"[base] Submission-cap mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}): {submission_cap_map:.5f}")


## 13. (추가 기능 1) SAM 박스 정제 — `USE_SAM`

검출 박스를 SAM의 box prompt로 넣어 얻은 마스크의 외접 박스로 좌표를 바꿉니다. 학습이 없고, val에서 padding(px)을 자동 튜닝합니다.
`sam2.1_b.pt`를 먼저 시도하고 실패하면 다음 후보로 넘어갑니다(첫 실행 시 가중치 자동 다운로드).


In [ ]:
# ══ SAM: 검출 박스 → 마스크 외접 박스로 정제 (USE_SAM) ══
# 검출 박스를 SAM의 box prompt로 넣어 마스크를 얻고, 마스크의 외접 박스로 좌표를 바꿉니다.
# 마스크가 이상하면(원본과 IoU 낮음 / 면적 너무 작음) 해당 박스는 원본을 그대로 유지합니다.
SAM_ACTIVE = False
sam_model = None
val_sam = None                       # stem -> (N,4) SAM 마스크 박스(정제 안 한 행은 NaN)
SAM_DEVICE = 0 if torch.cuda.is_available() else "cpu"
_sam_fail_count = 0


def _largest_component_box(mask_bool):
    """가장 큰 연결 성분의 외접 박스와 면적. 없으면 None."""
    n_labels, _, stats, _ = cv2.connectedComponentsWithStats(
        mask_bool.astype(np.uint8), connectivity=8
    )
    if n_labels <= 1:
        return None
    idx = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h, area = [int(v) for v in stats[idx]]
    return [float(x), float(y), float(x + w), float(y + h)], area


def sam_mask_boxes(image_path, boxes_xyxy, wh):
    """boxes (M,4) → (M,4) 마스크 외접 박스. 실패한 행은 NaN."""
    boxes = np.asarray(boxes_xyxy, dtype=np.float32).reshape(-1, 4)
    out = np.full((len(boxes), 4), np.nan, dtype=np.float32)
    if len(boxes) == 0:
        return out
    results = sam_model.predict(
        str(image_path), bboxes=boxes.tolist(), device=SAM_DEVICE, verbose=False
    )
    if not results or results[0].masks is None:
        return out
    masks = results[0].masks.data
    masks = masks.detach().cpu().numpy() if hasattr(masks, "detach") else np.asarray(masks)
    if masks.ndim == 2:
        masks = masks[None]
    if len(masks) != len(boxes):
        return out
    for i, mask in enumerate(masks):
        found = _largest_component_box(mask > 0.5)
        if found is None:
            continue
        box, area = found
        mask_h, mask_w = mask.shape[:2]
        sx, sy = wh[0] / mask_w, wh[1] / mask_h       # 마스크가 원본 해상도가 아닐 때 보정
        box = [box[0] * sx, box[1] * sy, box[2] * sx, box[3] * sy]
        box_area = max(float((boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])), 1.0)
        if area * sx * sy < SAM_MIN_AREA_RATIO * box_area:
            continue
        out[i] = box
    return out


def sam_boxes_for_pred(image_path, pred):
    """pred의 상위 SAM_TOPK개(점수 ≥ SAM_MIN_SCORE)에 대한 마스크 박스. 나머지는 NaN."""
    global _sam_fail_count
    n = len(pred["scores"])
    out = np.full((n, 4), np.nan, dtype=np.float32)
    idx = [i for i in range(min(n, SAM_TOPK)) if pred["scores"][i] >= SAM_MIN_SCORE]
    if not idx:
        return out
    boxes = clip_xyxy(pred["xyxy"][idx], pred["wh"])
    keep = (boxes[:, 2] - boxes[:, 0] > 1) & (boxes[:, 3] - boxes[:, 1] > 1)
    idx = [i for i, k in zip(idx, keep) if k]
    boxes = boxes[keep]
    if not idx:
        return out
    try:
        out[idx] = sam_mask_boxes(image_path, boxes, pred["wh"])
    except Exception as exc:                        # 한 장 실패가 전체를 멈추지 않게 함
        _sam_fail_count += 1
        if _sam_fail_count <= 3:
            print(f"⚠ SAM 추론 실패(원본 박스 유지): {type(exc).__name__}: {str(exc)[:150]}")
    return out


def apply_sam(pred, sam_xyxy, pad=None, blend=None, min_iou=None):
    """pred의 박스를 SAM 박스로 교체(조건 만족 행만). 새 pred dict를 반환."""
    pad = SAM_PAD_PX if pad is None else pad
    blend = SAM_BLEND if blend is None else blend
    min_iou = SAM_MIN_IOU if min_iou is None else min_iou
    new = pred["xyxy"].copy()
    valid = ~np.isnan(sam_xyxy).any(axis=1)
    if valid.any():
        sam_box = sam_xyxy[valid]
        orig = pred["xyxy"][valid]
        ref = sam_box + np.array([-pad, -pad, pad, pad], dtype=np.float32)
        mixed = clip_xyxy(blend * ref + (1.0 - blend) * orig, pred["wh"])
        good = (
            (box_iou_pairwise(sam_box, orig) >= min_iou)
            & (mixed[:, 2] - mixed[:, 0] > 1.0)
            & (mixed[:, 3] - mixed[:, 1] > 1.0)
        )
        rows = np.flatnonzero(valid)[good]
        new[rows] = mixed[good]
    out = dict(pred)
    out["xyxy"] = new
    return out


def setup_sam():
    global sam_model
    from ultralytics import SAM
    last_error = None
    for name in SAM_MODEL_CANDIDATES:
        try:
            sam_model = SAM(name)
            # 자체 점검: GT 박스를 prompt로 넣었을 때 마스크가 나오는지, GT 박스 스타일(타이트한지) 확인
            ious = []
            for stem in valid_stems[:20]:
                rec = image_records[stem]
                gt = np.array([[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in rec["annotations"])],
                              dtype=np.float32)
                got = sam_mask_boxes(val_paths[stem], gt, (rec["width"], rec["height"]))
                ok = ~np.isnan(got).any(axis=1)
                if ok.any():
                    ious.extend(box_iou_pairwise(got[ok], gt[ok]).tolist())
            if not ious:
                raise RuntimeError("자체 점검 실패: 마스크가 반환되지 않았습니다.")
            print(f"SAM 로드: {name} | GT박스 vs SAM마스크박스 평균 IoU = {np.mean(ious):.3f} "
                  f"(1.0에 가까울수록 GT 박스가 마스크에 타이트)")
            return name
        except Exception as exc:
            last_error = f"{name}: {type(exc).__name__}: {str(exc)[:150]}"
            print(f"  SAM 후보 실패 → {last_error}")
            sam_model = None
    raise RuntimeError(f"SAM 후보를 모두 로드하지 못했습니다. 마지막 오류: {last_error}")


if USE_SAM:
    try:
        setup_sam()
        val_sam = {}
        for stem in tqdm(valid_stems, desc="SAM 정제 (val)"):
            val_sam[stem] = sam_boxes_for_pred(val_paths[stem], val_raw[stem])
        n_refined = sum(int((~np.isnan(v).any(axis=1)).sum()) for v in val_sam.values())
        n_total = sum(min(len(v), SAM_TOPK) for v in val_sam.values())
        print(f"SAM 마스크 박스 확보: {n_refined:,} / {n_total:,}")
        if n_refined == 0:
            raise RuntimeError("SAM 마스크 박스를 하나도 얻지 못했습니다.")
        SAM_ACTIVE = True
    except Exception:
        import traceback
        traceback.print_exc()
        print("⚠ SAM 준비에 실패해 SAM 없이 진행합니다. (위 오류를 확인하세요)")
        SAM_ACTIVE = False
else:
    print("USE_SAM=False: SAM 단계를 건너뜁니다.")

if SAM_ACTIVE:
    def _sam_map(pad):
        return {s: apply_sam(val_raw[s], val_sam[s], pad=pad) for s in valid_stems}

    if SAM_TUNE_PAD:
        pad_rows = []
        for pad in SAM_PAD_GRID:
            pad_rows.append({"pad_px": float(pad), **evaluate_variant(_sam_map(pad), valid_stems, val_targets)})
        pad_table = pd.DataFrame(pad_rows)
        pad_table["abs_pad"] = pad_table["pad_px"].abs()
        display(pad_table.drop(columns="abs_pad"))
        best_row = pad_table.sort_values(["map8", "map20", "abs_pad"], ascending=[False, False, True]).iloc[0]
        SAM_PAD_PX = float(best_row["pad_px"])
        print(f"SAM_PAD_PX 자동 선택: {SAM_PAD_PX:+.1f}px (val 기준, 단일 스칼라 튜닝)")
    variant_scores["sam"] = evaluate_variant(_sam_map(SAM_PAD_PX), valid_stems, val_targets)
    print(f"[sam] mAP@[.75:.95] max20={variant_scores['sam']['map20']:.5f} | "
          f"max8={variant_scores['sam']['map8']:.5f}")

    def show_sam_compare(stem, top=4):
        import matplotlib.patches as patches
        pred = val_raw[stem]
        refined = apply_sam(pred, val_sam[stem])
        image = Image.open(val_paths[stem]).convert("RGB")
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(image)
        for i in range(min(top, len(pred["scores"]))):
            for box, color, style in ((pred["xyxy"][i], "red", "--"), (refined["xyxy"][i], "lime", "-")):
                ax.add_patch(patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                               fill=False, edgecolor=color, linewidth=1.5, linestyle=style))
        ax.set_title(f"{stem}  (빨강 점선=원본, 초록=SAM 정제)")
        ax.axis("off")
        plt.show()
        image.close()

    try:
        for sample_stem in random.sample(valid_stems, min(2, len(valid_stems))):
            show_sam_compare(sample_stem)
    except Exception as exc:
        print("SAM 비교 시각화 생략:", type(exc).__name__, exc)


## 14. (추가 기능 2) SimCLR crop 분류기 — `USE_SIMCLR`

train GT crop으로 대조학습(SimCLR) → 118종 분류 fine-tuning 후, 검출 박스를 crop해 클래스를 재판정하고 검출기 클래스와 융합합니다.
가중치는 `addons/` 폴더에 캐시되어 같은 split·설정이면 재사용됩니다.


In [ ]:
# ══ SimCLR crop 분류기 (USE_SIMCLR) ══
# 1) train split의 GT crop만으로 대조학습(SimCLR) → 2) 118종 분류 fine-tuning
# 3) 검출된 박스를 crop해 클래스 확률을 얻고, 검출기 클래스와 융합합니다.
# val/test 이미지는 학습에 쓰지 않습니다(누수 방지). GPU 위에서 증강해 DataLoader 없이 동작합니다.
import torch.nn as nn
import torch.nn.functional as F
import torchvision

SIMCLR_ACTIVE = False
simclr_net = None
SIMCLR_FUSE_ALPHA_FINAL = SIMCLR_FUSE_ALPHA
SIMCLR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_IMNET_MEAN = (0.485, 0.456, 0.406)
_IMNET_STD = (0.229, 0.224, 0.225)
if SIMCLR_DEVICE == "cuda" and torch.cuda.is_bf16_supported():
    SIMCLR_AMP_DTYPE = torch.bfloat16
elif SIMCLR_DEVICE == "cuda":
    SIMCLR_AMP_DTYPE = torch.float16
else:
    SIMCLR_AMP_DTYPE = None


def amp_ctx():
    if SIMCLR_AMP_DTYPE is None:
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=SIMCLR_AMP_DTYPE)


def make_scaler():
    if SIMCLR_AMP_DTYPE == torch.float16:
        return torch.amp.GradScaler("cuda")
    return None


def backward_step(loss, optimizer, scaler, params, clip=5.0):
    optimizer.zero_grad(set_to_none=True)
    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, clip)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, clip)
        optimizer.step()


# ── 모델 ─────────────────────────────────────────────────────
class PillNet(nn.Module):
    def __init__(self, backbone="resnet50", imagenet_init=True, proj_dim=128, num_classes=0):
        super().__init__()
        if not str(backbone).startswith("resnet"):
            raise ValueError("SIMCLR_BACKBONE은 resnet18 / resnet34 / resnet50 중에서 고르세요.")
        weights = None
        if imagenet_init:
            try:
                weights = torchvision.models.get_model_weights(backbone).DEFAULT
            except Exception as exc:
                print("ImageNet 가중치 정보를 찾지 못해 랜덤 초기화:", exc)
        try:
            net = torchvision.models.get_model(backbone, weights=weights)
        except Exception as exc:
            print(f"ImageNet 가중치를 받지 못해 랜덤 초기화로 진행: {type(exc).__name__}")
            net = torchvision.models.get_model(backbone, weights=None)
        self.feat_dim = int(net.fc.in_features)
        net.fc = nn.Identity()
        self.backbone = net
        self.proj = nn.Sequential(
            nn.Linear(self.feat_dim, 512), nn.ReLU(inplace=True), nn.Linear(512, proj_dim)
        )
        self.head = nn.Linear(self.feat_dim, num_classes) if num_classes > 0 else None
        self.register_buffer("mean", torch.tensor(_IMNET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(_IMNET_STD).view(1, 3, 1, 1), persistent=False)

    def features(self, x01):
        return self.backbone((x01 - self.mean) / self.std)

    def embed(self, x01):
        return F.normalize(self.proj(self.features(x01)).float(), dim=1)

    def classify(self, x01):
        return self.head(self.features(x01))


# ── crop · GPU 증강 ──────────────────────────────────────────
def crop_square(img, box, size, pad_ratio):
    """박스 중심 기준 정사각 crop(여백 포함, 이미지 밖은 검정) → size×size."""
    x1, y1, x2, y2 = [float(v) for v in box]
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(x2 - x1, y2 - y1, 8.0) * (1.0 + pad_ratio)
    s = max(int(round(side)), 2)
    left, top = int(round(cx - s / 2.0)), int(round(cy - s / 2.0))
    return img.crop((left, top, left + s, top + s)).resize((size, size), Image.BILINEAR)


def collect_gt_crops(stems, desc):
    total = sum(len(image_records[s]["annotations"]) for s in stems)
    size = SIMCLR_CROP
    crops = np.empty((total, size, size, 3), dtype=np.uint8)
    labels = np.empty(total, dtype=np.int64)
    k = 0
    for stem in tqdm(stems, desc=desc):
        record = image_records[stem]
        with Image.open(record["image_path"]) as opened:
            image = opened.convert("RGB")
        for ann in record["annotations"]:
            x, y, w, h = ann["bbox"]
            crops[k] = np.asarray(crop_square(image, (x, y, x + w, y + h), size, SIMCLR_CROP_PAD))
            labels[k] = code_to_model_id[ann["category_code"]]
            k += 1
        image.close()
    return crops, labels


def gpu_augment(x, out_size, scale=(0.4, 1.0), color=0.3, noise=0.02):
    """x: (B,3,H,W) float[0,1] → 랜덤 확대·이동·회전·반전 + 약한 색 변형. 알약은 색이 중요해 hue는 건드리지 않음."""
    batch, channels = x.shape[0], x.shape[1]
    dev = x.device
    zoom = torch.empty(batch, device=dev).uniform_(scale[0], scale[1]).sqrt()
    angle = torch.rand(batch, device=dev) * (2.0 * math.pi)
    flip = (torch.rand(batch, device=dev) < 0.5).float() * 2.0 - 1.0
    max_shift = 1.0 - zoom
    tx = (torch.rand(batch, device=dev) * 2.0 - 1.0) * max_shift
    ty = (torch.rand(batch, device=dev) * 2.0 - 1.0) * max_shift
    cos, sin = torch.cos(angle), torch.sin(angle)
    theta = torch.zeros(batch, 2, 3, device=dev)
    theta[:, 0, 0] = zoom * cos * flip
    theta[:, 0, 1] = -zoom * sin
    theta[:, 0, 2] = tx
    theta[:, 1, 0] = zoom * sin * flip
    theta[:, 1, 1] = zoom * cos
    theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, (batch, channels, out_size, out_size), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if color > 0:
        bright = 1.0 + (torch.rand(batch, 1, 1, 1, device=dev) * 2.0 - 1.0) * color
        contrast = 1.0 + (torch.rand(batch, 1, 1, 1, device=dev) * 2.0 - 1.0) * color
        satur = 1.0 + (torch.rand(batch, 1, 1, 1, device=dev) * 2.0 - 1.0) * color
        x = x * bright
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - mean) * contrast + mean
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        x = (x - gray) * satur + gray
    if noise > 0:
        x = x + torch.randn_like(x) * noise
    return x.clamp(0.0, 1.0)


def nt_xent(z, temperature):
    n = z.shape[0] // 2
    sim = (z @ z.t()) / temperature
    sim = sim.masked_fill(torch.eye(2 * n, dtype=torch.bool, device=z.device), float("-inf"))
    targets = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, targets)


def _cosine_with_warmup(warm_steps, total_steps, floor=0.01):
    def fn(step):
        warm = min(1.0, (step + 1) / max(1, warm_steps))
        progress = min(1.0, step / max(1, total_steps))
        return warm * (floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * progress)))
    return fn


# ── 학습 ─────────────────────────────────────────────────────
def pretrain_simclr(crops_np, batch):
    net = PillNet(SIMCLR_BACKBONE, SIMCLR_IMAGENET_INIT, num_classes=0).to(SIMCLR_DEVICE)
    if SIMCLR_EPOCHS <= 0:
        return net
    crops = torch.from_numpy(crops_np)                      # NHWC uint8 (메모리 공유)
    n = len(crops)
    batch = max(2, min(batch, n))
    steps_per_epoch = max(1, n // batch)
    total_steps = SIMCLR_EPOCHS * steps_per_epoch
    params = list(net.backbone.parameters()) + list(net.proj.parameters())
    optimizer = torch.optim.AdamW(params, lr=SIMCLR_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, _cosine_with_warmup(steps_per_epoch, total_steps)
    )
    scaler = make_scaler()
    net.train()
    for epoch in range(SIMCLR_EPOCHS):
        perm = torch.randperm(n)
        running, count = 0.0, 0
        for it in range(steps_per_epoch):
            idx = perm[it * batch:(it + 1) * batch]
            xb = crops[idx].to(SIMCLR_DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            with torch.no_grad():
                v1 = gpu_augment(xb, SIMCLR_CROP, scale=(0.35, 1.0), color=0.30)
                v2 = gpu_augment(xb, SIMCLR_CROP, scale=(0.35, 1.0), color=0.30)
            with amp_ctx():
                z = net.embed(torch.cat([v1, v2], dim=0))
            loss = nt_xent(z, SIMCLR_TEMP)
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
            backward_step(loss, optimizer, scaler, params)
            scheduler.step()
            running += float(loss.detach())
            count += 1
        print(f"[SimCLR 사전학습] epoch {epoch + 1:02d}/{SIMCLR_EPOCHS} | loss {running / max(1, count):.4f}")
    return net


@torch.no_grad()
def predict_probs(net, crops_nhwc, batch=256, tta=True):
    """crops (N,S,S,3) uint8 → (N, num_classes) softmax 확률(numpy)."""
    net.eval()
    outputs = []
    for start in range(0, len(crops_nhwc), batch):
        xb = torch.as_tensor(np.ascontiguousarray(crops_nhwc[start:start + batch]))
        xb = xb.to(SIMCLR_DEVICE).permute(0, 3, 1, 2).float().div_(255.0)
        views = [xb] + ([torch.rot90(xb, k, (2, 3)) for k in (1, 2, 3)] if tta else [])
        acc = 0.0
        for view in views:
            with amp_ctx():
                logits = net.classify(view)
            acc = acc + F.softmax(logits.float(), dim=1)
        outputs.append((acc / len(views)).cpu())
    if not outputs:
        return np.zeros((0, num_classes), dtype=np.float32)
    return torch.cat(outputs).numpy().astype(np.float32)


def finetune_classifier(net, tr_x, tr_y, va_x, va_y, batch):
    net.head = nn.Linear(net.feat_dim, num_classes).to(SIMCLR_DEVICE)
    crops = torch.from_numpy(tr_x)
    labels = torch.from_numpy(tr_y)
    n = len(crops)
    batch = max(2, min(batch, n))
    steps_per_epoch = max(1, n // batch)
    total_steps = SIMCLR_CLS_EPOCHS * steps_per_epoch
    groups = [
        {"params": list(net.backbone.parameters()), "lr": SIMCLR_CLS_LR_BACKBONE},
        {"params": list(net.head.parameters()), "lr": SIMCLR_CLS_LR_HEAD},
    ]
    all_params = groups[0]["params"] + groups[1]["params"]
    optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, _cosine_with_warmup(steps_per_epoch, total_steps)
    )
    scaler = make_scaler()
    for epoch in range(SIMCLR_CLS_EPOCHS):
        net.train()
        perm = torch.randperm(n)
        running, count = 0.0, 0
        for it in range(steps_per_epoch):
            idx = perm[it * batch:(it + 1) * batch]
            xb = crops[idx].to(SIMCLR_DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            yb = labels[idx].to(SIMCLR_DEVICE)
            with torch.no_grad():
                xb = gpu_augment(xb, SIMCLR_CROP, scale=(0.6, 1.0), color=0.12, noise=0.01)
            with amp_ctx():
                logits = net.classify(xb)
            loss = F.cross_entropy(logits.float(), yb, label_smoothing=0.1)
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
            backward_step(loss, optimizer, scaler, all_params)
            scheduler.step()
            running += float(loss.detach())
            count += 1
        message = f"[SimCLR 분류 fine-tune] epoch {epoch + 1:02d}/{SIMCLR_CLS_EPOCHS} | loss {running / max(1, count):.4f}"
        if va_x is not None and len(va_x) and ((epoch + 1) % 4 == 0 or epoch + 1 == SIMCLR_CLS_EPOCHS):
            acc = float((predict_probs(net, va_x, tta=False).argmax(1) == va_y).mean())
            message += f" | val GT-crop acc {acc:.4f}"
        print(message)
    return net


def train_simclr_classifier(tr_x, tr_y, va_x, va_y):
    batch = SIMCLR_BATCH
    while True:
        try:
            net = pretrain_simclr(tr_x, batch)
            net = finetune_classifier(net, tr_x, tr_y, va_x, va_y, batch)
            return net
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if batch <= 16:
                raise
            batch //= 2
            print(f"⚠ GPU 메모리 부족 → SIMCLR 배치를 {batch}로 줄여 처음부터 다시 시도합니다.")


def setup_simclr():
    global simclr_net
    torch.manual_seed(SEED)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    train_hash = hashlib.sha1(",".join(train_stems).encode("utf-8")).hexdigest()[:10]
    ckpt_path = ADDON_DIR / (
        f"simclr_{SIMCLR_BACKBONE}_s{SIMCLR_CROP}_e{SIMCLR_EPOCHS}_c{SIMCLR_CLS_EPOCHS}_{train_hash}.pt"
    )
    net = None
    if SIMCLR_REUSE_CACHE and ckpt_path.exists():
        try:
            state = torch.load(ckpt_path, map_location="cpu")
            if state["sorted_codes"] != sorted_codes:
                raise ValueError("클래스 구성이 달라 캐시를 사용할 수 없습니다.")
            net = PillNet(SIMCLR_BACKBONE, False, num_classes=num_classes)
            net.load_state_dict(state["state_dict"])
            net.to(SIMCLR_DEVICE).eval()
            print(f"SimCLR 분류기 캐시 로드: {ckpt_path.name}")
        except Exception as exc:
            print(f"캐시 로드 실패 → 새로 학습합니다: {type(exc).__name__}: {str(exc)[:120]}")
            net = None
    va_x = va_y = None
    if net is None:
        tr_x, tr_y = collect_gt_crops(train_stems, "GT crop 수집 (train)")
        va_x, va_y = collect_gt_crops(valid_stems, "GT crop 수집 (val, 정확도 확인용)")
        print(f"train crops={len(tr_x):,}, val crops={len(va_x):,}, crop={SIMCLR_CROP}px, backbone={SIMCLR_BACKBONE}")
        net = train_simclr_classifier(tr_x, tr_y, va_x, va_y)
        net.eval()
        torch.save({
            "state_dict": net.state_dict(),
            "sorted_codes": sorted_codes,
            "backbone": SIMCLR_BACKBONE,
            "crop": SIMCLR_CROP,
        }, ckpt_path)
        print(f"저장: {ckpt_path}")
        del tr_x, tr_y
    if va_x is None:
        va_x, va_y = collect_gt_crops(valid_stems, "GT crop 수집 (val, 정확도 확인용)")
    acc = float((predict_probs(net, va_x, tta=SIMCLR_TTA).argmax(1) == va_y).mean())
    print(f"SimCLR 분류기 val GT-crop 정확도 (TTA={SIMCLR_TTA}): {acc:.4f}")
    simclr_net = net
    return net


# ── 검출 결과에 적용 ─────────────────────────────────────────
def simclr_probs_for_map(pred_map, stems, path_lookup, desc="SimCLR crop 분류"):
    """stem -> (K,C) 확률. 각 이미지에서 점수 상위 SIMCLR_TOPK개 중 점수 ≥ SIMCLR_MIN_SCORE인 선두 행만."""
    out, buf_crops, buf_keys = {}, [], []

    def flush():
        if not buf_crops:
            return
        probs = predict_probs(simclr_net, np.stack(buf_crops), tta=SIMCLR_TTA)
        for (stem_key, row), prob in zip(buf_keys, probs):
            out[stem_key][row] = prob
        buf_crops.clear()
        buf_keys.clear()

    for stem in tqdm(list(stems), desc=desc):
        pred = pred_map[stem]
        count = int((pred["scores"][:SIMCLR_TOPK] >= SIMCLR_MIN_SCORE).sum())
        out[stem] = np.zeros((count, num_classes), dtype=np.float32)
        if count == 0:
            continue
        with Image.open(path_lookup[stem]) as opened:
            image = opened.convert("RGB")
        for row in range(count):
            crop = crop_square(image, pred["xyxy"][row], SIMCLR_CROP, SIMCLR_CROP_PAD)
            buf_crops.append(np.asarray(crop))
            buf_keys.append((stem, row))
        image.close()
        if len(buf_crops) >= 256:
            flush()
    flush()
    return out


def fuse_simclr(pred, probs, alpha=None, topn=None):
    """검출 클래스 one-hot과 분류기 확률을 섞어 클래스 후보 top-N을 같은 박스에 출력.
    q = (1-alpha)*onehot(검출 클래스) + alpha*분류확률,  최종 점수 = 검출 점수 × q[class]"""
    alpha = SIMCLR_FUSE_ALPHA_FINAL if alpha is None else alpha
    topn = SIMCLR_TOPN if topn is None else topn
    boxes, scores, labels = [], [], []
    for i in range(len(pred["scores"])):
        if i < len(probs):
            q = alpha * probs[i]
            det_label = int(pred["labels"][i])
            if 0 <= det_label < num_classes:
                q[det_label] += 1.0 - alpha
            for cls in np.argsort(-q)[:topn]:
                if q[cls] <= 1e-6:
                    continue
                boxes.append(pred["xyxy"][i])
                scores.append(float(pred["scores"][i]) * float(q[cls]))
                labels.append(int(cls))
        else:
            boxes.append(pred["xyxy"][i])
            scores.append(float(pred["scores"][i]))
            labels.append(int(pred["labels"][i]))
    return make_pred(np.asarray(boxes).reshape(-1, 4), scores, labels, pred["wh"])


if USE_SIMCLR:
    try:
        setup_simclr()
        SIMCLR_ACTIVE = True
    except Exception:
        import traceback
        traceback.print_exc()
        print("⚠ SimCLR 준비에 실패해 SimCLR 없이 진행합니다. (위 오류를 확인하세요)")
        SIMCLR_ACTIVE = False
else:
    print("USE_SIMCLR=False: SimCLR 단계를 건너뜁니다.")


## 15. 변형별 val 점수 비교 · 제출 변형 결정


In [ ]:
# ══ 변형별 val 점수 비교 + 제출 변형 결정 ══
# base / sam / simclr / sam+simclr 중 켜져 있는 조합만 계산합니다.
val_maps = {"base": val_raw}
val_probs = {}

if SAM_ACTIVE:
    val_maps["sam"] = {s: apply_sam(val_raw[s], val_sam[s]) for s in valid_stems}

if SIMCLR_ACTIVE:
    try:
        val_probs["base"] = simclr_probs_for_map(val_raw, valid_stems, val_paths, desc="SimCLR crop 분류 (val, 원본 박스)")
        if SAM_ACTIVE:
            val_probs["sam"] = simclr_probs_for_map(val_maps["sam"], valid_stems, val_paths, desc="SimCLR crop 분류 (val, SAM 박스)")

        # 실제 제출에 쓰일 박스 기준으로 융합 비율(alpha) 선택
        tune_key = "sam" if SAM_ACTIVE else "base"
        if SIMCLR_TUNE_ALPHA:
            alpha_rows = []
            for alpha in SIMCLR_ALPHA_GRID:
                fused = {s: fuse_simclr(val_maps[tune_key][s], val_probs[tune_key][s], alpha=alpha) for s in valid_stems}
                alpha_rows.append({"alpha": float(alpha), **evaluate_variant(fused, valid_stems, val_targets)})
            alpha_table = pd.DataFrame(alpha_rows)
            display(alpha_table)
            SIMCLR_FUSE_ALPHA_FINAL = float(alpha_table.sort_values(["map8", "map20"], ascending=False).iloc[0]["alpha"])
            print(f"SimCLR 융합 alpha 자동 선택: {SIMCLR_FUSE_ALPHA_FINAL} (val 기준)")
        else:
            SIMCLR_FUSE_ALPHA_FINAL = float(SIMCLR_FUSE_ALPHA)

        val_maps["simclr"] = {s: fuse_simclr(val_raw[s], val_probs["base"][s]) for s in valid_stems}
        if SAM_ACTIVE:
            val_maps["sam+simclr"] = {s: fuse_simclr(val_maps["sam"][s], val_probs["sam"][s]) for s in valid_stems}
    except Exception:
        import traceback
        traceback.print_exc()
        print("⚠ SimCLR 융합 평가 중 오류 → SimCLR 없이 진행합니다.")
        SIMCLR_ACTIVE = False
        val_maps.pop("simclr", None)
        val_maps.pop("sam+simclr", None)

for name, pred_map in val_maps.items():
    if name not in variant_scores:
        variant_scores[name] = evaluate_variant(pred_map, valid_stems, val_targets)

comparison = pd.DataFrame(variant_scores).T.rename(columns={
    "map20": f"mAP@[.75:.95] (max {EVAL_MAX_DETECTIONS})",
    "map8": f"mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}, 제출 조건)",
})
comparison["Δ vs base (제출 조건)"] = (
    comparison[f"mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}, 제출 조건)"]
    - comparison.loc["base", f"mAP@[.75:.95] (max {SUBMISSION_MAX_DETECTIONS}, 제출 조건)"]
)
display(comparison.style.format("{:.5f}"))

FINAL_VARIANT = "+".join(
    name for name, on in (("sam", SAM_ACTIVE), ("simclr", SIMCLR_ACTIVE)) if on
) or "base"
if SUBMIT_USE_BEST_VARIANT:
    FINAL_VARIANT = max(variant_scores, key=lambda k: (variant_scores[k]["map8"], variant_scores[k]["map20"]))
print(f"USE_SAM={USE_SAM} (실제 적용={SAM_ACTIVE}) | USE_SIMCLR={USE_SIMCLR} (실제 적용={SIMCLR_ACTIVE})")
print(f"제출에 사용할 변형: {FINAL_VARIANT}  (val {variant_scores[FINAL_VARIANT]['map8']:.5f})")
if FINAL_VARIANT != "base" and variant_scores[FINAL_VARIANT]["map8"] < variant_scores["base"]["map8"]:
    print("⚠ 선택된 변형의 val 점수가 base보다 낮습니다. 스위치를 끄거나 SUBMIT_USE_BEST_VARIANT=True를 고려하세요.")


## 13. validation 예측 시각화


In [ ]:
def show_prediction(stem, threshold=.20, figsize=(10, 8)):
    image = Image.open(image_records[stem]["image_path"]).convert("RGB")
    result = best_model.predict(str(image_records[stem]["image_path"]), imgsz=IMGSZ, conf=threshold, max_det=EVAL_MAX_DETECTIONS, device=0, verbose=False)[0]
    boxes = result.boxes
    det = sv.Detections(xyxy=boxes.xyxy.cpu().numpy(), confidence=boxes.conf.cpu().numpy(), class_id=boxes.cls.cpu().numpy().astype(int))
    labels = [f"{model_id_to_code[int(c)]} {s:.3f}" for c,s in zip(det.class_id, det.confidence)]
    canvas = sv.BoxAnnotator(thickness=3).annotate(np.asarray(image).copy(), det)
    canvas = sv.LabelAnnotator(text_scale=.45).annotate(canvas, det, labels)
    plt.figure(figsize=figsize); plt.imshow(canvas); plt.title(f"Prediction · {stem}"); plt.axis("off"); plt.show(); image.close()

for sample_stem in random.sample(valid_stems, min(4, len(valid_stems))): show_prediction(sample_stem)


## 16. 테스트 추론 · 제출 CSV 생성


In [ ]:
# ══ 테스트 추론 → 변형 적용 → 제출 CSV ══
SUBMISSION_PATH = RUN_OUTPUT_DIR / f"submission_{SUBMISSION_TAG}_{FINAL_VARIANT.replace('+', '_')}.csv"

test_lookup = {p.stem: p for p in test_paths}
test_stems = [p.stem for p in test_paths]

test_raw = {}
for path_batch in tqdm(list(batched(test_paths, PRED_BATCH_SIZE)), desc="Test 검출"):
    for path, pred in zip(path_batch, detect_batch(path_batch, max_det=EVAL_MAX_DETECTIONS)):
        test_raw[path.stem] = pred

final_parts = FINAL_VARIANT.split("+")
test_pred = test_raw
if "sam" in final_parts:
    test_pred = {
        s: apply_sam(test_raw[s], sam_boxes_for_pred(test_lookup[s], test_raw[s]))
        for s in tqdm(test_stems, desc="SAM 정제 (test)")
    }
if "simclr" in final_parts:
    test_probs = simclr_probs_for_map(test_pred, test_stems, test_lookup, desc="SimCLR crop 분류 (test)")
    test_pred = {s: fuse_simclr(test_pred[s], test_probs[s]) for s in test_stems}

rows, annotation_id = [], 1
for stem in test_stems:
    try:
        image_id = int(stem)
    except ValueError as exc:
        raise ValueError(f"test filename stem이 숫자가 아닙니다: {stem!r}") from exc
    pred = test_pred[stem]
    boxes = clip_xyxy(pred["xyxy"], pred["wh"])
    for i in range(min(len(pred["scores"]), SUBMISSION_MAX_DETECTIONS)):
        x1, y1, x2, y2 = [float(v) for v in boxes[i]]
        bw, bh = x2 - x1, y2 - y1
        class_id = int(pred["labels"][i])
        if bw <= 0 or bh <= 0 or not (0 <= class_id < num_classes):
            continue
        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": code_to_submission_id[model_id_to_code[class_id]],
            "bbox_x": round(x1, 4),
            "bbox_y": round(y1, 4),
            "bbox_w": round(bw, 4),
            "bbox_h": round(bh, 4),
            "score": round(float(min(max(pred["scores"][i], 0.0), 1.0)), 6),
        })
        annotation_id += 1

submission_columns = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
submission = pd.DataFrame(rows, columns=submission_columns)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")
print(f"변형: {FINAL_VARIANT}")
print(f"저장 완료: {SUBMISSION_PATH.resolve()} / rows={len(submission):,}")
display(submission.head())


## 17. 제출 파일 검증


In [ ]:
assert submission.columns.tolist() == submission_columns
assert submission["annotation_id"].is_unique
assert (submission["bbox_w"] > 0).all() and (submission["bbox_h"] > 0).all()
assert submission["score"].between(0, 1).all()
assert set(submission["category_id"]).issubset(set(code_to_submission_id.values()))

test_image_ids = {int(path.stem) for path in test_paths}
prediction_counts = submission.groupby("image_id").size()
missing_prediction_ids = test_image_ids - set(prediction_counts.index)
print(f"제출 대상 이미지 수: {len(test_image_ids):,}")
print(f"예측 포함 이미지 수: {prediction_counts.size:,}")
print(f"예측 0개 test 이미지: {len(missing_prediction_ids)}")
display(prediction_counts.describe())
if missing_prediction_ids: raise RuntimeError(f"예측이 0개인 test 이미지: {sorted(missing_prediction_ids)[:20]}")


## 운영 메모 (v2)

1. 처음에는 `USE_SAM=False, USE_SIMCLR=False`로 기준선(base)을 만들고, 그 다음 스위치를 하나씩 켜서 비교표를 확인하세요.
2. RTX 5060에서 OOM이면 `BATCH=2`(NBS=64 유지). 여유가 있어도 최종 비교 전에는 모델·해상도를 바꾸지 마세요.
3. 모델 비교는 Ultralytics 기본 COCO mAP가 아니라 `mAP@[0.75:0.95]`(비교표의 제출 조건 열)를 기준으로 합니다.
4. 수동 중단 후에는 `RESUME=True`, `RUN_TRAINING=False`로 바꾸어 같은 `last.pt`에서 이어갑니다.
5. 학습을 건너뛰고 기존 결과로 평가·제출만 하려면 `RUN_TRAINING=False`와 `EXISTING_RUN_DIR`를 지정하세요.
6. SAM/SimCLR는 val로 스칼라(padding, alpha)를 튜닝하므로 val 점수가 약간 낙관적일 수 있습니다. 최종 판단은 리더보드 점수와 함께 하세요.
